# Recurrent neural networks

For di module wey we do before now, we don talk about rich semantic representations of text. Di architecture we dey use dey capture di combined meaning of words for one sentence, but e no dey consider di **order** of di words, because di aggregation operation wey dey follow di embeddings go comot dis information from di original text. Becos dis kain models no fit represent word ordering, dem no fit solve more complex or unclear tasks like text generation or question answering.

To capture di meaning of text sequence, we go use one neural network architecture wey dem dey call **recurrent neural network**, or RNN. When we dey use RNN, we dey pass our sentence through di network one token at a time, and di network dey produce some **state**, we go carry dat state give di network again with di next token.

![Image showing an example recurrent neural network generation.](../../../../../translated_images/pcm/rnn.27f5c29c53d727b5.webp)

Given di input sequence of tokens $X_0,\dots,X_n$, di RNN dey create sequence of neural network blocks, and e go train dis sequence end-to-end using backpropagation. Each network block dey take pair $(X_i,S_i)$ as input, and e produce $S_{i+1}$ as result. Di final state $S_n$ or output $Y_n$ go enter one linear classifier to produce di result. All di network blocks share di same weights, and dem dey trained end-to-end using one back propagation pass.

> Di figure wey dey above show recurrent neural network for unrolled form (on top left), and for more compact recurrent representation (on top right). E important to realize say all RNN Cells get di same **shareable weights**.

Becos state vectors $S_0,\dots,S_n$ dey passed through di network, di RNN fit learn sequential dependencies between words. For example, when word *not* show somewhere for di sequence, e fit learn to negate some elements inside di state vector.

Inside, each RNN cell get two weight matrices: $W_H$ and $W_I$, plus bias $b$. For each RNN step, if dem give input $X_i$ and input state $S_i$, output state go calculate as $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, where $f$ na activation function (wey most times be $\tanh$).

> For problems like text generation (wey we go cover for di next unit) or machine translation, we sef want get some output value for each RNN step. For dis kain case, another matrix go dey $W_O$, and output go calculate as $Y_i=f(W_O\times S_i+b_O)$.

Make we see how recurrent neural networks fit help us classify our news dataset.

> For di sandbox environment, we need run di next cell to make sure di required library don install, and data don prefetch. If you dey run am locally, you fit skip di next cell.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Wen you dey train big big models, GPU memory fit turn wahala. We fit also need try different minibatch sizes, so dat di data go fit inside our GPU memory, but di training go still fast. If you dey run dis code for your own GPU machine, you fit try adjust minibatch size to make training quick.

> **Note**: Some versions of NVidia drivers no dey release memory after dem don train di model. We dey run plenty examples inside dis notebook, and e fit make memory finish for some setups, especially if you dey do your own experiments for di same notebook. If you see some strange errors wen you start to train di model, e good make you restart di notebook kernel.


In [3]:
batch_size = 16
embed_size = 64

## Simple RNN classifier

For simple RNN mata, each recurrent unit na simple linear network, wey dey take input vector and state vector, come give new state vector. For Keras, e fit represent by the `SimpleRNN` layer.

Even though we fit pass one-hot encoded tokens straight to RNN layer, e no too good because their dimensionality high. So, we go use embedding layer to reduce the dimensionality of word vectors, come follow am with RNN layer, come end with `Dense` classifier.

> **Note**: For cases we dimensionality no too high, like when you dey use character-level tokenization, e fit make sense to pass one-hot encoded tokens direct inside the RNN cell.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Note:** Wi dey use untrained embedding layer here for simplicity, but for beta results we fit use pretrained embedding layer wit Word2Vec, like how e bin talk for di previous unit. E go good make you try adjust dis code make e fit work wit pretrained embeddings.

Now make we train our RNN. RNN dem no too easy to train, because once RNN cells dem don unroll along sequence length, di amount of layers wey go dey for backpropagation go plenty. So we gatz choose smaller learning rate, and train di network on bigger dataset make e give beta results. Dis fit take long time, so to use GPU go better.

To make am quick, we go only train di RNN model on news titles, no go include di description. You fit try train wit description and see if you fit make di model train.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Note** se accuracy fit be lower for here, because we dey train only on news titles.


## Revisiting variable sequences 

Membaa say di `TextVectorization` layer go automatically add pad tokens for sequences wey get different length for one minibatch. E turn out say dem pad tokens sef dey join for training, an e fit make model no fit learn well.

We get different ways we fit take reduce how much padding we dey use. One way na to arrange di dataset according to sequence length, den group all sequence dem for size. You fit do dis one wit di `tf.data.experimental.bucket_by_sequence_length` function (check [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)).

Another way na to use **masking**. For Keras, some layers get extra input wey fit show which tokens we suppose use for training. To put masking for inside our model, we fit add one separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we fit just put di `mask_zero=True` parameter for our `Embedding` layer.

> **Note**: Dis training go carry like 5 minutes to finish one epoch for whole dataset. No wahala to stop training any time if you no get patience again. Wetin you fit do sef na to limit how much data you dey use for training, by adding `.take(...)` after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Now we don dey use masking, we fit train di model ontop di whole dataset of titles and descriptions.

> **Note**: You don notice say we don dey use vectorizer wey dem train ontop di news titles, no di whole body of di article? E fit be say som tokens no go fit dey consider, so e good to train di vectorizer again. But e fit be say e no go too affect well well, so we go still use di previous pre-trained vectorizer to make am simple.


## LSTM: Long short-term memory

One of di main wahala wey RNNs get na **vanishing gradients**. RNN fit long well well, and e fit hard for am to carry di gradients go all di way back to di first layer of di network during backpropagation. When dis one happen, di network no fit learn relationship between tokens wey far. One way to avoid dis wahala na to introduce **explicit state management** by using **gates**. Di two common architecture wey de use gates na **long short-term memory** (LSTM) and **gated relay unit** (GRU). We go talk LSTM here.

![Image showing an example long short term memory cell](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

One LSTM network arrange like RNN, but e get two states wey dem dey pass from layer to layer: di actual state $c$, and di hidden vector $h$. For each unit, di hidden vector $h_{t-1}$ join with input $x_t$, and dem control wetin go happen to di state $c_t$ and output $h_{t}$ through **gates**. Each gate get sigmoid activation (output wey de range $[0,1]$), we fit think am as bitwise mask when e multiply di state vector. LSTMs get dis gates (from left to right for di picture wey de above):
* **forget gate** wey dey decide which components for di vector $c_{t-1}$ we gats forget, and which one we gats pass through.
* **input gate** wey dey decide how much information from di input vector and previous hidden vector go enter di state vector.
* **output gate** wey dey take di new state vector and decide which of di parts go use to produce di new hidden vector $h_t$.

Di parts of di state $c$ fit be like flags wey fit turn on and off. For example, when we see di name *Alice* for di sequence, we go guess say e mean woman, and we go raise di flag for di state wey talk say we get female noun for di sentence. When we also see di words *and Tom*, we go raise di flag say we get plural noun. So by manipulating state, we fit dey track di grammatical properties of di sentence.

> **Note**: Dis na correct resource to sabi how inside LSTMs dey work: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) by Christopher Olah.

Although di inside structure of LSTM cell fit look complex, Keras dey hide dis implementation inside di `LSTM` layer, so di only tin we need do for di example wey de above na to replace di recurrent layer:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Note** se train LSTMs sef dey slow, an e fit be say you no go see much beta for accuracy for di beginning of training. You fit need continue train for some time to get better accuracy.


## Bidirectional an multilayer RNN dem

For our examples we don do so far, di recurrent networks dey work from di beginning of sequence go till e finish. Dis one dey normal for us because e follow di same direction wey we dey read or listen to speech. But, for sometimes wey e need make we fit run random check for di input sequence, e make sense make di recurrent computation run for both directions. RNN dem wey fit run computations for both directions na dem dey call **bidirectional** RNNs, an we fit create dem by wrapping di recurrent layer wit a special `Bidirectonal` layer.

> **Note**: Di `Bidirectional` layer go make two copies of di layer inside am, an e go set di `go_backwards` property for one of those copies to `True`, wey go make am run for di opposite direction along di sequence.

Recurrent networks, whether na unidirectional or bidirectional, dey capture patterns inside sequence, an dem dey store am inside state vectors or dem fit return am as output. Like how convolutional networks dey work, we fit build another recurrent layer wey go follow di first one to capture higher level patterns, wey dey build from lower level patterns wey di first layer extract. Dis one bring us to di idea of **multi-layer RNN**, wey get two or more recurrent networks, wey di output from di previous layer na im e be input for di next layer.

![Image showing a Multilayer long-short-term-memory- RNN](../../../../../translated_images/pcm/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Picture from [dis wonderful post](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) by Fernando López.*

Keras make am easy to build dis kind networks, because you just need to add more recurrent layers to di model. For all di layers except di last one, we need to add di `return_sequences=True` parameter, because we need di layer to return all di intermediate states, no be only di final state of di recurrent computation.

Make we build two-layer bidirectional LSTM for our classification wahala.

> **Note** dis code again go take time before e finish, but e go give us di highest accuracy we don see so far. So e fit worth to wait and see di result.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs for oda tasks

Up until now, we don focus on to use RNNs for classify text sequence dem. But dem fit do plenty oda job, like text generation and machine translation &mdash; we go look those tasks for di next unit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
